## Imports

In [1]:
import numpy as np
from sklearn.metrics import adjusted_rand_score, accuracy_score, normalized_mutual_info_score, v_measure_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import torch

## Synthetic Data

#### Data generator

In [2]:
from experiments.synthetic_datasets import generate_synthetic_data_with_probs
from sklearn.metrics import adjusted_rand_score, accuracy_score, normalized_mutual_info_score, v_measure_score
from model.training import majority_vote_relabeling, majority_vote_relabeling_full

### K-Means Classifier

In [14]:
# --- Helper Function for Maximal Accuracy (Label Matching) ---
def cluster_acc_max(y_true, y_pred):
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)
    
    # Create the contingency matrix
    labels = np.unique(y_true)
    preds = np.unique(y_pred)
    C = np.zeros((len(labels), len(preds)), dtype=int)
    
    for i in range(len(labels)):
        for j in range(len(preds)):
            C[i, j] = np.sum((y_true == labels[i]) & (y_pred == preds[j]))

    # Find the maximum assignment cost (correct matches)
    row_ind, col_ind = linear_sum_assignment(-C)
    
    return C[row_ind, col_ind].sum() / len(y_true)
    

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, accuracy_score, normalized_mutual_info_score, v_measure_score
from scipy.optimize import linear_sum_assignment
from model.training import majority_vote_relabeling
from experiments.calibration import conformal_prediction_kmeans_with_synthetic_data
from experiments.synthetic_training import train_autoencoder_e1_synthetic_data
from torch.utils.data import Dataset, DataLoader, TensorDataset


#### Gen data --> Compress to lower dimension --> Predict clusters

def kmeans_cluster(X_train, X_test):  
    kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
    y_train_pred = kmeans.fit_predict(X_train)
    y_test_pred = kmeans.predict(X_test)
    return kmeans, y_train_pred, y_test_pred

input_dim = 512
latent_dim = 32
batch_size=64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def gen_data():
    xy_train, xy_calib, xy_test, p0, p1 = generate_synthetic_data_with_probs(N_train=1426,dim=input_dim, mean0=0,std0=0.3,mean1=0.1,std1=0)
    ## 10000 = 14246
    ## 1000 = 1426
    ## 5000 = 7124
    X_train = torch.tensor(xy_train[0], dtype=torch.float32)
    y_train_true = xy_train[1]
    X_test = torch.tensor(xy_test[0], dtype=torch.float32)
    y_test_true = xy_test[1]
    X_cal = torch.tensor(xy_calib[0], dtype=torch.float32)
    y_cal_true = xy_calib[1]
    return X_train, y_train_true, X_test, y_test_true, X_cal, y_cal_true


### Dataset and loaders
X_train, y_train_true, X_test, y_test_true, X_cal, y_cal_true = gen_data()
train_dataset = TensorDataset(X_train, torch.tensor(y_train_true, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=batch_size)

## AE
ae_model = train_autoencoder_e1_synthetic_data(train_loader, input_dim=input_dim, latent_dim=latent_dim, device = DEVICE, epochs=5)


### Predict

aris, nmis, v_measures, accuracys = [], [], [], []
list_q_hat, list_emp_marginal_coverage, list_avg_pred_set_size, list_set_sizes_per_class, list_cc_per_class = [], [], [], [], []
NUM_ITERS = 5
kmeans_cluster_mapping = None
for i in range(0, NUM_ITERS):
    X_train, y_train_true, X_test, y_test_true, X_cal, y_cal_true = gen_data()

    ## Compress
    X_train_latent = ae_model.encoder(X_train.to(DEVICE)).detach().cpu().numpy()
    X_test_latent = ae_model.encoder(X_test.to(DEVICE)).detach().cpu().numpy()
    X_cal_latent = ae_model.encoder(X_cal.to(DEVICE)).detach().cpu().numpy()

    total_elements = X_train_latent.shape[0]
    indices_1d = torch.randperm(total_elements)
    
    X_train_latent = X_train_latent[indices_1d]
    y_train_true = y_train_true[indices_1d]
    
    kmeans_model, y_train_pred, y_test_pred = kmeans_cluster(X_train_latent,X_test_latent)
    _, kmeans_cluster_mapping = majority_vote_relabeling(y_train_pred, y_train_true)

    # If there are more than one label:
    #if len(np.unique(y_test_pred)) > 1:
    y_test_pred_relab =  np.vectorize(kmeans_cluster_mapping.get)(y_test_pred)
    #else:
    #    y_test_pred_relab = y_test_pred
        
    accuracy = accuracy_score(y_test_true, y_test_pred_relab)

    ## Set sizes
    prediction_sets, y_calib_pred_relab, y_val_pred_relab, res_metrics = \
    conformal_prediction_kmeans_with_synthetic_data(kmeans_model, (X_cal_latent,y_cal_true), (X_test_latent,y_test_true), 
                                                    kmeans_cluster_mapping, alpha=0.1, print_stats=False)

    ##############################   Results
    list_q_hat.append(res_metrics[0])
    list_emp_marginal_coverage.append(res_metrics[1])
    list_avg_pred_set_size.append(res_metrics[2])
    list_set_sizes_per_class.append(res_metrics[3])
    list_cc_per_class.append(res_metrics[4])
    aris.append(round(adjusted_rand_score(y_test_true, y_test_pred_relab),3))
    nmis.append(round(normalized_mutual_info_score(y_test_true, y_test_pred_relab, average_method='geometric'),3))
    v_measures.append(round(v_measure_score(y_test_true, y_test_pred_relab),3))
    accuracys.append(round(accuracy_score(y_test_true, y_test_pred_relab),3))


results = {
    "list_q_hat" : list_q_hat,
    "list_emp_marginal_coverage": list_emp_marginal_coverage,
    "list_avg_pred_set_size": list_avg_pred_set_size,
    "list_set_sizes_per_class": list_set_sizes_per_class,
    "list_cc_per_class": list_cc_per_class,
    "list_ari": aris,
    "list_nmi": nmis,
    "list_v_measure": v_measures,
    "list_accuracy": accuracys
}


results_kmeans_syn_df = pd.DataFrame(results)



#### Conformal Set-sizes

In [208]:
from experiments.calibration import conformal_prediction_knn_with_synthetic_data

In [ ]:
prediction_sets, y_calib_pred_relab, y_val_pred_relab, res_metrics = \
    conformal_prediction_kmeans_with_synthetic_data(kmeans_model, xy_calib, xy_test, 
                                                    kmeans_cluster_mapping, alpha=0.1, print_stats=False)


In [829]:
def conv_results(res_metrics):
    new_res_metrics = {}

    list_q_hat_0 = []
    list_q_hat_1 = []
    for v in res_metrics[0]:
        if 0 in v:
            list_q_hat_0.append(np.round(v[0], 3).item())
        if 1 in v:   
            list_q_hat_1.append(np.round(v[1], 3).item())
    new_res_metrics['q_hat_class0'] = list_q_hat_0
    new_res_metrics['q_hat_class1'] = list_q_hat_1

    list_mc = []
    for v in res_metrics[1]:
        w = round(v, 3)
        list_mc.append(w)
    new_res_metrics['empirical_marginal_cov'] = (', '.join(map(str, list_mc)))

    list_apss = []
    for v in res_metrics[2]:
        list_apss.append(np.round(v, 3).item())
    new_res_metrics['avg_pred_set_size'] = list_apss

    list_pred_set_sizes_class0 = []
    list_pred_set_sizes_class1 = []
    list_pred_set_sizes_class2 = []
    for v in res_metrics[3]:
        if 0 in v:
            list_pred_set_sizes_class0.append(np.round(v[0], 3).item())
        if 1 in v:
            list_pred_set_sizes_class1.append(np.round(v[1], 3).item())
        if 2 in v:
            list_pred_set_sizes_class2.append(np.round(v[2], 3).item())
    new_res_metrics['pred_set_sizes_class0'] = list_pred_set_sizes_class0 if len(list_pred_set_sizes_class0)>0 else []
    new_res_metrics['pred_set_sizes_class1'] = list_pred_set_sizes_class1 if len(list_pred_set_sizes_class1)>0 else []
    new_res_metrics['pred_set_sizes_class2'] = list_pred_set_sizes_class2 if len(list_pred_set_sizes_class2)>0 else []

    list_class0_cov = []
    list_class1_cov = []
    for v in res_metrics[4]:
        if 0 in v:
            list_class0_cov.append(np.round(v[0], 3).item())
        if 1 in v:
            list_class1_cov.append(np.round(v[1], 3).item())
    new_res_metrics['class0_cov'] = list_class0_cov
    new_res_metrics['class1_cov'] = list_class1_cov

    return new_res_metrics
        
    

In [15]:
##print(conv_results(res_metrics))

### KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import adjusted_rand_score, accuracy_score, normalized_mutual_info_score, v_measure_score
from scipy.optimize import linear_sum_assignment
from model.training import majority_vote_relabeling
from experiments.calibration import conformal_prediction_knn_with_synthetic_data
from experiments.synthetic_training import train_autoencoder_e1_synthetic_data
from torch.utils.data import Dataset, DataLoader, TensorDataset


def knn_cluster(X_train, y_train, X_test):
    # k-NN is trained on the labeled data (X_train, y_train)
    k_neighbors = 5 # Standard choice for k
    knn_classifier = KNeighborsClassifier(n_neighbors=k_neighbors)
    knn_classifier.fit(X_train, y_train) # Training utilizes y_train
    y_train_pred = knn_classifier.predict(X_train)
    y_test_pred = knn_classifier.predict(X_test)
    return knn_classifier, y_train_pred, y_test_pred

input_dim = 512
latent_dim = 32
batch_size=64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def gen_data():
    xy_train, xy_calib, xy_test, p0, p1 = generate_synthetic_data_with_probs(N_train=1426,dim=input_dim, mean0=0,std0=0.3,mean1=0.05,std1=0)
    ## 10000 = 14246
    ## 1000 = 1426
    ## 5000 = 7124
    X_train = torch.tensor(xy_train[0], dtype=torch.float32)
    y_train_true = xy_train[1]
    X_test = torch.tensor(xy_test[0], dtype=torch.float32)
    y_test_true = xy_test[1]
    X_cal = torch.tensor(xy_calib[0], dtype=torch.float32)
    y_cal_true = xy_calib[1]
    return X_train, y_train_true, X_test, y_test_true, X_cal, y_cal_true


### Dataset and loaders
X_train, y_train_true, X_test, y_test_true, X_cal, y_cal_true = gen_data()
train_dataset = TensorDataset(X_train, torch.tensor(y_train_true, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=batch_size)

## AE
ae_model = train_autoencoder_e1_synthetic_data(train_loader, input_dim=input_dim, latent_dim=latent_dim, device = DEVICE, epochs=5)


### Predict

aris, nmis, v_measures, accuracys = [], [], [], []
list_q_hat, list_emp_marginal_coverage, list_avg_pred_set_size, list_set_sizes_per_class, list_cc_per_class = [], [], [], [], []
NUM_ITERS = 5
kmeans_cluster_mapping = None
for i in range(0, NUM_ITERS):
    X_train, y_train_true, X_test, y_test_true, X_cal, y_cal_true = gen_data()

    ## Compress
    X_train_latent = ae_model.encoder(X_train.to(DEVICE)).detach().cpu().numpy()
    X_test_latent = ae_model.encoder(X_test.to(DEVICE)).detach().cpu().numpy()
    X_cal_latent = ae_model.encoder(X_cal.to(DEVICE)).detach().cpu().numpy()

    total_elements = X_train_latent.shape[0]
    indices_1d = torch.randperm(total_elements)
    
    X_train_latent = X_train_latent[indices_1d]
    y_train_true = y_train_true[indices_1d]
    
    knn_model,y_train_pred, y_test_pred = knn_cluster(X_train_latent,y_train_true,X_test_latent)
    
    y_test_pred_relab =  y_test_pred
        
    accuracy = accuracy_score(y_test_true, y_test_pred_relab)

    ## Set sizes
    prediction_sets, y_calib_pred_relab, y_val_pred_relab, res_metrics = \
    conformal_prediction_knn_with_synthetic_data(knn_model, (X_cal_latent,y_cal_true), (X_test_latent,y_test_true), 
                                                    alpha=0.1, print_stats=False)

    ##############################   Results
    list_q_hat.append(res_metrics[0])
    list_emp_marginal_coverage.append(res_metrics[1])
    list_avg_pred_set_size.append(res_metrics[2])
    list_set_sizes_per_class.append(res_metrics[3])
    list_cc_per_class.append(res_metrics[4])
    aris.append(round(adjusted_rand_score(y_test_true, y_test_pred_relab),3))
    nmis.append(round(normalized_mutual_info_score(y_test_true, y_test_pred_relab, average_method='geometric'),3))
    v_measures.append(round(v_measure_score(y_test_true, y_test_pred_relab),3))
    accuracys.append(round(accuracy_score(y_test_true, y_test_pred_relab),3))


results = {
    "list_q_hat" : list_q_hat,
    "list_emp_marginal_coverage": list_emp_marginal_coverage,
    "list_avg_pred_set_size": list_avg_pred_set_size,
    "list_set_sizes_per_class": list_set_sizes_per_class,
    "list_cc_per_class": list_cc_per_class,
    "list_ari": aris,
    "list_nmi": nmis,
    "list_v_measure": v_measures,
    "list_accuracy": accuracys
}


results_knn_syn_df = pd.DataFrame(results)


### ContraSD

In [ ]:
from experiments.synthetic_datasets import setup_synthetic_dataloaders,generate_synthetic_data_with_probs
from experiments.synthetic_training import extract_embeddings_synthetic

## Dataloaders

# 10000 = 14246
# 8000 = 11397
# 5000 = 7124
# 1000 = 1426
# 100 = 144

input_dim = 512
latent_dim = 32
batch_size = 64
train_set, calib_set, val_set, p0, p1 = generate_synthetic_data_with_probs(N_train=1426,dim=input_dim, mean0=0,std0=0.3,mean1=0.2,std1=0.0)
contrastive_train_loader_syn,train_loader_syn,calib_loader_syn,val_loader_syn = setup_synthetic_dataloaders(train_set, calib_set, val_set, batch_size)



In [ ]:
from experiments.synthetic_training import train_autoencoder_e2_synthetic_data, train_contrastive_embeddings_transformer_with_synthetic_data
import torch

# MLP Projector
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs=100
emb_model = train_contrastive_embeddings_transformer_with_synthetic_data(contrastive_train_loader_syn, input_dim, 
                                                           latent_dim, batch_size=batch_size, device=device, epochs=epochs)

In [ ]:
# Train autoencoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 5
ae_basic_model = \
    train_autoencoder_e2_synthetic_data(train_loader=train_loader_syn, embed_transformer=emb_model, input_dim=input_dim, latent_dim=latent_dim, device=device, epochs=epochs)



In [ ]:
from model.models import ContrastiveEmbeddingsTransformer
from experiments.eval_models import ContrastiveClusterLearnerExt, EmbeddingsAutoencoderServiceExt
from experiments.ethics_training import conformal_prediction_with_ethics_data, CustomException   



def gen_data(input_dim=input_dim):
    xy_train, xy_calib, xy_test, p0, p1 = generate_synthetic_data_with_probs(N_train=1426,dim=input_dim, mean0=0,std0=0.3,mean1=0.2,std1=0)
    ## 10000 = 14246
    ## 1000 = 1426
    ## 5000 = 7124
    X_train = torch.tensor(xy_train[0], dtype=torch.float32)
    y_train_true = xy_train[1]
    X_test = torch.tensor(xy_test[0], dtype=torch.float32)
    y_test_true = xy_test[1]
    X_cal = torch.tensor(xy_calib[0], dtype=torch.float32)
    y_cal_true = xy_calib[1]
    return X_train, y_train_true, X_test, y_test_true, X_cal, y_cal_true

## Transform data for clustering

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cse_model = EmbeddingsAutoencoderServiceExt(autoencoder=ae_basic_model, embed_transformer=emb_model, device=DEVICE)
ccl_model = ContrastiveClusterLearnerExt(embed_transformer = cse_model)


########

aris, nmis, v_measures, accuracys = [], [], [], []
list_q_hat, list_emp_marginal_coverage, list_avg_pred_set_size, list_set_sizes_per_class, list_cc_per_class = [], [], [], [], []
NUM_ITERS = 5
input_dim = train_set[0].shape[1]
for i in range(0, NUM_ITERS):
    #############################  Permute
    X_train, y_train_true, X_test, y_test_true, X_cal, y_cal_true = gen_data(input_dim)
    X_train = torch.tensor(train_set[0], dtype=torch.float32).to(DEVICE)
    y_train_true = train_set[1]
    X_test = torch.tensor(val_set[0], dtype=torch.float32).to(DEVICE)
    y_test_true = val_set[1]
    X_cal = torch.tensor(calib_set[0], dtype=torch.float32).to(DEVICE)
    y_cal_true = calib_set[1]
    
    ## Transform embeddings
    X_train_latent = cse_model.encode_transform(X_train)
    X_cal_latent = cse_model.encode_transform(X_cal)
    X_test_latent = cse_model.encode_transform(X_test)

    try:
        ccl_model, prediction_sets, centroids_train, clusters_labels_mapping, pred_calib_labels, pred_val_labels, res_metrics = \
        conformal_prediction_with_ethics_data(ccl_model, X_train_latent, y_train_true, X_cal_latent,
                                                X_test_latent, DEVICE, alpha=0.1, print_stats=False)

    except CustomException as ce:
        print(f"Logging exception: {ce}")
        continue

    ##############################  Predict
    y_test_pred_relab = pred_val_labels.detach().cpu()

    
    ##############################   Results
    list_q_hat.append(res_metrics[0])
    list_emp_marginal_coverage.append(res_metrics[1])
    list_avg_pred_set_size.append(res_metrics[2])
    list_set_sizes_per_class.append(res_metrics[3])
    list_cc_per_class.append(res_metrics[4])
    aris.append(round(adjusted_rand_score(y_test_true, y_test_pred_relab),3))
    nmis.append(round(normalized_mutual_info_score(y_test_true, y_test_pred_relab, average_method='geometric'),3))
    v_measures.append(round(v_measure_score(y_test_true, y_test_pred_relab),3))
    accuracys.append(round(accuracy_score(y_test_true, y_test_pred_relab),3))


results = {
    "list_q_hat" : list_q_hat,
    "list_emp_marginal_coverage": list_emp_marginal_coverage,
    "list_avg_pred_set_size": list_avg_pred_set_size,
    "list_set_sizes_per_class": list_set_sizes_per_class,
    "list_cc_per_class": list_cc_per_class,
    "list_ari": aris,
    "list_nmi": nmis,
    "list_v_measure": v_measures,
    "list_accuracy": accuracys
}


results_contrasd_syn_df = pd.DataFrame(results)


## ETHICS Data

### Generate training and calibration sets

In [113]:
from experiments.ethics_contrastive_dataset import setup_ethics_contrastive_training

NUM_PAIRS=3000
BATCH_SIZE=64

contrastive_train_dataloader,train_labels,milvus_dataset = setup_ethics_contrastive_training(num_pairs_per_type=NUM_PAIRS, batch_size=BATCH_SIZE)


Total entities: 2520
Number of train entities: 2016
Number of validation entities: 252
Number of calibration entities: 252
2520 = 2016 + 252 + 252 = 2520
Generated 6000 pairs (3000 similar) for training.


#### Test Data

In [3]:
from experiments.ethics import EthicsDataset
import torch

collection_name = "ethics_virtues_test_dataset_embeddings_qwen38b"
ds = EthicsDataset(db_name='word_sense_embeddings',batch_size=64)
res = ds.get_items(collection=collection_name, num_items=500)

embeddings_new = torch.empty(0)
labels_new = []

for r in res:
    if embeddings_new.numel() == 0:
        embeddings_new = torch.tensor(r['embedding'])
    else:
        embeddings_new = torch.vstack((embeddings_new, torch.tensor(r['embedding'])))
    labels_new.append(r['label'])

X_test = embeddings_new
y_test = np.array(labels_new)

#### Training Data for Model Evaluation

In [35]:
from experiments.ethics_contrastive_dataset import setup_ethics_contrastive_training, fetch_subset, extract_from_dataloader

## Train data: FOR TRAINING MODELS OTHER THAN CONTRASD FOR COMPARISON.

train_dataset_dataloader = fetch_subset(milvus_dataset, 'train')
inputs_train, embeddings_train, labels_train = extract_from_dataloader(train_dataset_dataloader)

print(f"Number of inputs in train set: {len(inputs_train)}")
print(f"Shape of embeddings in train set: {embeddings_train.shape}")
print(f"Number of labels in train set: {len(labels_train)}")

print(f"Sample item in input list: {inputs_train[0]}")

Number of inputs in train set: 2016
Shape of embeddings in train set: torch.Size([2016, 4096])
Number of labels in train set: 2016
Sample item in input list: (462833194181506020, 'truthful: When lucas was asked about the broken window, he told his mom he broke it on accident.')


In [36]:
calib_dataset_dataloader = fetch_subset(milvus_dataset, 'calibration')
inputs_calib, embeddings_calib, labels_calib = extract_from_dataloader(calib_dataset_dataloader)

In [37]:
### Training set

X_train = embeddings_train
y_train = np.array(labels_train)


In [31]:
import numpy as np

def generate_bootstrap_samples(X, y, n_bootstrap_samples=1000):
    """
    Generates bootstrap samples (X_boot, y_boot) by sampling indices with replacement.

    Args:
        X (np.ndarray): Data tensor with shape (N_samples, N_features).
        y (np.ndarray): Labels tensor with shape (N_samples,).
        n_bootstrap_samples (int): The number of bootstrap samples to generate.

    Returns:
        list: A list of n_bootstrap_samples tuples, where each tuple contains
              (X_boot, y_boot) for one resampled dataset.
    """
    N_samples = X.shape[0]
    bootstrap_datasets = []

    # Array of original indices: [0, 1, 2, ..., 99]
    original_indices = np.arange(N_samples)

    for i in range(n_bootstrap_samples):
        # 1. Sample N_samples indices WITH replacement
        # This is the core of the bootstrapping technique.
        # The resulting array will contain 100 indices, with some duplicated and some omitted.
        sampled_indices = np.random.choice(original_indices, size=N_samples, replace=True)

        # 2. Use the sampled indices to create the bootstrap sample
        X_boot = X[sampled_indices]
        y_boot = y[sampled_indices]

        bootstrap_datasets.append((X_boot, y_boot))
        
    return bootstrap_datasets

### ContraSD

In [ ]:
from experiments.ethics_contrastive_dataset import fetch_subset, extract_from_dataloader

calib_dataset_dataloader = fetch_subset(milvus_dataset, 'calibration')
inputs_calib, embeddings_calib, labels_calib = extract_from_dataloader(calib_dataset_dataloader)

print(f"Number of words in calibration set: {len(inputs_calib)}")
print(f"Shape of embeddings in calibration set: {embeddings_calib.shape}")
print(f"Number of labels in calibration set: {len(labels_calib)}")

print(f"Sample item in input list: {inputs_calib[0]}")

In [ ]:
validation_dataset_dataloader=fetch_subset(milvus_dataset, 'validation')
inputs_valid, embeddings_valid, labels_valid = extract_from_dataloader(validation_dataset_dataloader)

print(f"Number of inputs in validation set: {len(inputs_valid)}")
print(f"Shape of embeddings in validation set: {embeddings_valid.shape}")
print(f"Number of labels in validation set: {len(labels_valid)}")

print(f"Sample item in input list: {inputs_valid[0]}")

#### Training

In [ ]:
## Train

from model.training import train_autoencoder_prepared_data, train_contrastive_embeddings_transformer_prepared_data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_DIM = 4096
EMBED_DIM = 256
BATCH_SIZE = 64
EPOCHS = 80
LEARNING_RATE = 0.0001
MARGIN = 0.5

emb_model = train_contrastive_embeddings_transformer_prepared_data(contrastive_train_loader=contrastive_train_dataloader, input_dim=INPUT_DIM, embedding_dim = EMBED_DIM, 
                                                     device = DEVICE, epochs=EPOCHS, learning_rate=LEARNING_RATE, margin=MARGIN)



In [ ]:
from experiments.ethics_training import train_autoencoder_ethics_data

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_DIM = 4096
LATENT_DIM = 256
BATCH_SIZE = 64
EPOCHS = 4
LEARNING_RATE = 0.001

ae_model, train_dataloader = train_autoencoder_ethics_data(emb_model, milvus_dataset, input_dim=INPUT_DIM, latent_dim=LATENT_DIM, 
                                                             device = DEVICE, batch_size=BATCH_SIZE, epochs=EPOCHS, learning_rate=LEARNING_RATE)



#### Calibration and Inference

In [56]:
from model.datasets import MilvusDataset
from torch.utils.data import DataLoader, Dataset


DB_NAME = 'word_sense_embeddings'
BATCH_SIZE = 64
SHUFFLE = False # Shuffle data every epoch
NUM_WORKERS = 0 # Use multi-process data loading (optional, set to 0 for single process)

COLLECTION_NAME = 'ethics_virtues_train_dataset_embeddings_qwen38b'
ethics_train_dataset = MilvusDataset(
    db_name=DB_NAME,
    collection_name=COLLECTION_NAME,
    batch_size = BATCH_SIZE,
    output_fields = ["id", "embedding", "label", "input"]
)
ethics_train_dataloader = DataLoader(
    ethics_train_dataset,
    batch_size=None,
    shuffle=SHUFFLE,
    num_workers=NUM_WORKERS
)
COLLECTION_NAME = 'ethics_virtues_test_dataset_embeddings_qwen38b'
ethics_test_dataset = MilvusDataset(
    db_name=DB_NAME,
    collection_name=COLLECTION_NAME,
    batch_size = BATCH_SIZE,
    output_fields = ["id", "embedding", "label", "input"]
)
ethics_test_dataloader = DataLoader(
    ethics_test_dataset,
    batch_size=None,
    shuffle=SHUFFLE,
    num_workers=NUM_WORKERS
)

#### Test with Ethics Data

In [ ]:
from model.models import ContrastiveEmbeddingsTransformer
from experiments.eval_models import ContrastiveClusterLearnerExt, EmbeddingsAutoencoderServiceExt
from experiments.ethics_training import conformal_prediction_with_ethics_data, CustomException

## Transform data for clustering

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cse_model = EmbeddingsAutoencoderServiceExt(autoencoder=ae_model, embed_transformer=emb_model, device=DEVICE)
ccl_model = ContrastiveClusterLearnerExt(embed_transformer = cse_model)

train_emb, inputs, labels_train = ccl_model.transform_ethics_embeddings(ethics_train_dataloader)
print(f"Training data shape: {train_emb.shape}")
val_emb = cse_model.encode_transform(embeddings_valid)
print(f"Validation data shape: {val_emb.shape}")
calib_emb = cse_model.encode_transform(embeddings_calib)
print(f"Calibration data shape: {calib_emb.shape}")

y_test_true = y_test

## Transform test embeddings
X_test_latent = cse_model.encode_transform(X_test)
print(f"Test data size: {X_test_latent.shape}")
print(f"Test labels size: {y_test_true.shape}")

aris, nmis, v_measures, accuracys = [], [], [], []
list_q_hat, list_emp_marginal_coverage, list_avg_pred_set_size, list_set_sizes_per_class, list_cc_per_class = [], [], [], [], []
NUM_ITERS = 5
for i in range(0, NUM_ITERS):
    #############################  Permute
    total_elements = train_emb.shape[0]
    indices_1d = torch.randperm(total_elements)
    
    X_train_latent = train_emb[indices_1d]
    y_train = labels_train[indices_1d]

    try:
        ccl_model, prediction_sets, centroids_train, clusters_labels_mapping, pred_calib_labels, pred_val_labels, res_metrics = \
        conformal_prediction_with_ethics_data(ccl_model, X_train_latent, y_train, calib_emb,
                                                X_test_latent, DEVICE, alpha=0.1, print_stats=False)
    except CustomException as ce:
        print(f"Logging exception: {ce}")
        continue

    ##############################  Predict
    y_test_pred = ccl_model.predict_clusters(X_test_latent, centroids_train)
    y_test_pred = y_test_pred.detach().cpu()
    y_test_pred_relab = np.vectorize(clusters_labels_mapping.get)(y_test_pred)

    
    ##############################   Results
    list_q_hat.append(res_metrics[0])
    list_emp_marginal_coverage.append(res_metrics[1])
    list_avg_pred_set_size.append(res_metrics[2])
    list_set_sizes_per_class.append(res_metrics[3])
    list_cc_per_class.append(res_metrics[4])
    aris.append(adjusted_rand_score(y_test_true, y_test_pred_relab))
    nmis.append(normalized_mutual_info_score(y_test_true, y_test_pred_relab, average_method='geometric'))
    v_measures.append(v_measure_score(y_test_true, y_test_pred_relab))
    accuracys.append(accuracy_score(y_test_true, y_test_pred_relab))


results = {
    "list_q_hat" : list_q_hat,
    "list_emp_marginal_coverage": list_emp_marginal_coverage,
    "list_avg_pred_set_size": list_avg_pred_set_size,
    "list_set_sizes_per_class": list_set_sizes_per_class,
    "list_cc_per_class": list_cc_per_class,
    "list_ari": aris,
    "list_nmi": nmis,
    "list_v_measure": v_measures,
    "list_accuracy": accuracys
}


contrasd_results_df = pd.DataFrame(results)

print(conv_results(contrasd_results_df))

### SBERT

#### Dataset

In [ ]:
from experiments.ethics_contrastive_dataset import setup_ethics_contrastive_training

NUM_PAIRS=10
BATCH_SIZE=64

db_name='word_sense_embeddings'
collection_name='ethics_virtues_train_dataset_embeddings_allminilm'
contrastive_train_dataloader,train_labels,milvus_dataset = setup_ethics_contrastive_training(
    db_name=db_name,
    collection_name=collection_name,
    num_pairs_per_type=NUM_PAIRS, 
    batch_size=BATCH_SIZE
)


In [ ]:
from experiments.ethics_contrastive_dataset import fetch_subset, extract_from_dataloader

calib_dataset_dataloader = fetch_subset(milvus_dataset, 'calibration')
inputs_calib, embeddings_calib, labels_calib = extract_from_dataloader(calib_dataset_dataloader)

print(f"Number of words in calibration set: {len(inputs_calib)}")
print(f"Shape of embeddings in calibration set: {embeddings_calib.shape}")
print(f"Number of labels in calibration set: {len(labels_calib)}")

print(f"Sample item in input list: {inputs_calib[0]}")

In [ ]:
validation_dataset_dataloader=fetch_subset(milvus_dataset, 'validation')
inputs_valid, embeddings_valid, labels_valid = extract_from_dataloader(validation_dataset_dataloader)

print(f"Number of inputs in validation set: {len(inputs_valid)}")
print(f"Shape of embeddings in validation set: {embeddings_valid.shape}")
print(f"Number of labels in validation set: {len(labels_valid)}")

print(f"Sample item in input list: {inputs_valid[0]}")

In [ ]:
train_dataset_dataloader = fetch_subset(milvus_dataset, 'train')
inputs_train, embeddings_train, labels_train = extract_from_dataloader(train_dataset_dataloader)

print(f"Number of inputs in train set: {len(inputs_train)}")
print(f"Shape of embeddings in train set: {embeddings_train.shape}")
print(f"Number of labels in train set: {len(labels_train)}")

print(f"Sample item in input list: {inputs_train[0]}")

In [6]:
from experiments.ethics import EthicsDataset
import torch

collection_name = "ethics_virtues_test_dataset_embeddings_allminilm"
ds = EthicsDataset(db_name='word_sense_embeddings',batch_size=64)
res = ds.get_items(collection=collection_name, num_items=500)

embeddings_new = torch.empty(0)
labels_new = []

for r in res:
    if embeddings_new.numel() == 0:
        embeddings_new = torch.tensor(r['embedding'])
    else:
        embeddings_new = torch.vstack((embeddings_new, torch.tensor(r['embedding'])))
    labels_new.append(r['label'])

X_test = embeddings_new
y_test = np.array(labels_new)

#### Train AE with an Encoder that maps 384 dimensionality data for comparison

In [7]:
embeddings_train.shape

torch.Size([2016, 384])

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SbertOutputAutoencoder(nn.Module):
    def __init__(self, feature_dim, latent_dim=16):
        super(SbertOutputAutoencoder, self).__init__()
        self.latent_dim = latent_dim
        self.feature_dim = feature_dim
        self.encoder = nn.Sequential(
            nn.Linear(self.feature_dim, self.latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.feature_dim),
            nn.ReLU(),
        )
    def forward(self, x):
        latent = self.encoder(x)
        decoded = self.decoder(latent)
        return decoded

In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim

train_dataset = TensorDataset(embeddings_train)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 4
LEARNING_RATE = 0.001

model = SbertOutputAutoencoder(feature_dim=384, latent_dim=128)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model.train()
model.to(DEVICE)
for epoch in range(EPOCHS):
    for i, batch in enumerate(train_dataloader):
        inputs = batch[0]      # TensorDataset itself returns a tuple or a list of items for each sample
        inputs = inputs.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")


In [21]:
from sentence_transformers import SentenceTransformer
from model.models import ContrastiveEmbeddingsTransformer, ContrastiveClusterLearner
from experiments.eval_models import ContrastiveClusterLearnerExt, EmbeddingsAutoencoderServiceExt
from experiments.ethics_training import conformal_prediction_with_ethics_data, CustomException

## Transform data for clustering

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sbert_model_name = '/home/jovyan/sbert_cache/saved/all-MiniLM-L6-v2' # Or "all-MiniLM-L6-v2"
emb_model = SentenceTransformer(sbert_model_name)

cse_model = EmbeddingsAutoencoderServiceExt(autoencoder=model, embed_transformer=emb_model, device=DEVICE)
ccl_model_tf = ContrastiveClusterLearnerExt(embed_transformer = cse_model)
ccl_model = ContrastiveClusterLearner(embed_transformer = emb_model, device=DEVICE)


In [11]:
from model.datasets import MilvusDataset
from torch.utils.data import DataLoader, Dataset


DB_NAME = 'word_sense_embeddings'
BATCH_SIZE = 64
SHUFFLE = False # Shuffle data every epoch
NUM_WORKERS = 0 # Use multi-process data loading (optional, set to 0 for single process)

COLLECTION_NAME = 'ethics_virtues_train_dataset_embeddings_allminilm'
ethics_train_dataset = MilvusDataset(
    db_name=DB_NAME,
    collection_name=COLLECTION_NAME,
    batch_size = BATCH_SIZE,
    output_fields = ["id", "embedding", "label", "input"]
)
ethics_train_dataloader = DataLoader(
    ethics_train_dataset,
    batch_size=None,
    shuffle=SHUFFLE,
    num_workers=NUM_WORKERS
)
COLLECTION_NAME = 'ethics_virtues_test_dataset_embeddings_allminilm'
ethics_test_dataset = MilvusDataset(
    db_name=DB_NAME,
    collection_name=COLLECTION_NAME,
    batch_size = BATCH_SIZE,
    output_fields = ["id", "embedding", "label", "input"]
)
ethics_test_dataloader = DataLoader(
    ethics_test_dataset,
    batch_size=None,
    shuffle=SHUFFLE,
    num_workers=NUM_WORKERS
)

In [12]:
def conv_results(results_df):
    new_res_metrics = {}

    if "list_q_hat" in results_df:
        list_q_hat_0 = []
        list_q_hat_1 = []
        for v in results_df["list_q_hat"].tolist():
            if 0 in v:
                list_q_hat_0.append(v[0])
            if 1 in v:   
                list_q_hat_1.append(v[1])
        new_res_metrics['q_hat_class0'] = [x.item() for x in np.round(list_q_hat_0,3)]
        new_res_metrics['q_hat_class1'] = [x.item() for x in np.round(list_q_hat_1,3)]
    
        new_res_metrics['empirical_marginal_cov'] = [x.item() for x in np.round(results_df['list_emp_marginal_coverage'].tolist(),3)]
        new_res_metrics['avg_pred_set_size'] = [x.item() for x in np.round(results_df['list_avg_pred_set_size'].tolist(),3)]
    
    # These are always present
    new_res_metrics['ari'] = [x.item() for x in np.round(results_df['list_ari'].tolist(),3)]
    new_res_metrics['nmi'] = [x.item() for x in np.round(results_df['list_nmi'].tolist(),3)]
    new_res_metrics['v_measure'] = [x.item() for x in np.round(results_df['list_v_measure'].tolist(),3)]
    new_res_metrics['accuracy'] = [x.item() for x in np.round(results_df['list_accuracy'].tolist(),3)]
    

    if "list_set_sizes_per_class" in results_df:
        list_pred_set_sizes_class0 = []
        list_pred_set_sizes_class1 = []
        list_pred_set_sizes_class2 = []
        for v in results_df['list_set_sizes_per_class']:
            if 0 in v:
                list_pred_set_sizes_class0.append(np.round(v[0], 3).item())
            if 1 in v:
                list_pred_set_sizes_class1.append(np.round(v[1], 3).item())
            if 2 in v:
                list_pred_set_sizes_class2.append(np.round(v[2], 3).item())
        new_res_metrics['pred_set_sizes_class0'] = list_pred_set_sizes_class0 if len(list_pred_set_sizes_class0)>0 else []
        new_res_metrics['pred_set_sizes_class1'] = list_pred_set_sizes_class1 if len(list_pred_set_sizes_class1)>0 else []
        new_res_metrics['pred_set_sizes_class2'] = list_pred_set_sizes_class2 if len(list_pred_set_sizes_class2)>0 else []
    
        list_class0_cov = []
        list_class1_cov = []
        for v in results_df['list_cc_per_class']:
            if 0 in v:
                list_class0_cov.append(np.round(v[0], 3).item())
            if 1 in v:
                list_class1_cov.append(np.round(v[1], 3).item())
        new_res_metrics['class0_cov'] = list_class0_cov
        new_res_metrics['class1_cov'] = list_class1_cov

    return new_res_metrics
        

In [ ]:
train_emb, inputs, labels_train = ccl_model_tf.transform_ethics_embeddings(ethics_train_dataloader)
print(f"Training data shape: {train_emb.shape}")
val_emb = cse_model.encode_transform(embeddings_valid)
print(f"Validation data shape: {val_emb.shape}")
calib_emb = cse_model.encode_transform(embeddings_calib)
print(f"Calibration data shape: {calib_emb.shape}")

y_test_true = y_test

## Transform test embeddings
X_test_latent = cse_model.encode_transform(X_test)
print(f"Test data size: {X_test_latent.shape}")
print(f"Test labels size: {y_test_true.shape}")

aris, nmis, v_measures, accuracys = [], [], [], []
list_q_hat, list_emp_marginal_coverage, list_avg_pred_set_size, list_set_sizes_per_class, list_cc_per_class = [], [], [], [], []
NUM_ITERS = 5
for i in range(0, NUM_ITERS):
    #############################  Permute
    total_elements = train_emb.shape[0]
    indices_1d = torch.randperm(total_elements)
    
    X_train_latent = train_emb[indices_1d]
    y_train = labels_train[indices_1d]

    try:
        ccl_model, prediction_sets, centroids_train, clusters_labels_mapping, pred_calib_labels, pred_val_labels, res_metrics = \
        conformal_prediction_with_ethics_data(ccl_model, X_train_latent, y_train, calib_emb,
                                                X_test_latent, DEVICE, alpha=0.1, print_stats=False)
    except CustomException as ce:
        print(f"Logging exception: {ce}")
        continue

    ##############################  Predict
    y_test_pred = ccl_model.predict_clusters(X_test_latent, centroids_train)
    y_test_pred = y_test_pred.detach().cpu()
    y_test_pred_relab = np.vectorize(clusters_labels_mapping.get)(y_test_pred)

    
    ##############################   Results
    list_q_hat.append(res_metrics[0])
    list_emp_marginal_coverage.append(res_metrics[1])
    list_avg_pred_set_size.append(res_metrics[2])
    list_set_sizes_per_class.append(res_metrics[3])
    list_cc_per_class.append(res_metrics[4])
    aris.append(adjusted_rand_score(y_test_true, y_test_pred_relab))
    nmis.append(normalized_mutual_info_score(y_test_true, y_test_pred_relab, average_method='geometric'))
    v_measures.append(v_measure_score(y_test_true, y_test_pred_relab))
    accuracys.append(accuracy_score(y_test_true, y_test_pred_relab))


results = {
    "list_q_hat" : list_q_hat,
    "list_emp_marginal_coverage": list_emp_marginal_coverage,
    "list_avg_pred_set_size": list_avg_pred_set_size,
    "list_set_sizes_per_class": list_set_sizes_per_class,
    "list_cc_per_class": list_cc_per_class,
    "list_ari": aris,
    "list_nmi": nmis,
    "list_v_measure": v_measures,
    "list_accuracy": accuracys
}

contrasd_results_df = pd.DataFrame(results)

print(conv_results(contrasd_results_df))

### K-Means

In [38]:
X_train.shape

torch.Size([2016, 4096])

#### Prepare data

In [39]:
from experiments.eval_models import AutoencoderE1
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
import torch.optim as optim
import numpy as np

def train_autoencoder_for_model_comparison(X_orig, input_dim=4096, latent_dim=128, device = "cpu", epochs=10):
    DEVICE = device
    FEATURE_DIM = input_dim
    LATENT_DIM = latent_dim
    EPOCHS = epochs
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001

    model = AutoencoderE1(feature_dim=FEATURE_DIM, latent_dim=LATENT_DIM).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Dataloader
    train_dataset = TensorDataset(X_orig)
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # Training Loop 
    print(f"--- Starting Training on {DEVICE} ---")
    
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch in train_dataloader:
            # Input data is the target for an autoencoder (X = Y)
            inputs = batch[0]
            inputs = inputs.to(DEVICE) 
            
            # Forward pass
            outputs = model(inputs)
            
            # Loss calculation (output vs. original data)
            loss = criterion(outputs, inputs)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

    print("\nTraining Complete.")
    return model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
basic_ae = train_autoencoder_for_model_comparison(X_train, latent_dim=128, device = device, epochs=10)

In [ ]:
X_train_latent = basic_ae.encoder(X_train.to(device)).detach().cpu().numpy()
print(X_train_latent.shape)
X_test_latent = basic_ae.encoder(X_test.to(device)).detach().cpu().numpy()
print(X_test_latent.shape)
X_calib_latent = basic_ae.encoder(embeddings_calib.to(device)).detach().cpu().numpy()
print(X_calib_latent.shape)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, accuracy_score, normalized_mutual_info_score, v_measure_score
from model.training import majority_vote_relabeling
from experiments.calibration import conformal_prediction_kmeans_with_synthetic_data

def kmeans_cluster(X_train, X_test):  
    kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
    y_train_pred = kmeans.fit_predict(X_train)
    y_test_pred = kmeans.predict(X_test)
    return kmeans, y_train_pred, y_test_pred



y_test_true = y_test    # Ethics

aris, nmis, v_measures, accuracys = [], [], [], []
list_q_hat, list_emp_marginal_coverage, list_avg_pred_set_size, list_set_sizes_per_class, list_cc_per_class = [], [], [], [], []
NUM_ITERS = 10
kmeans_cluster_mapping = None
for i in range(0, NUM_ITERS):
    # Random selections
    n = 2000
    total_elements = X_train_latent.shape[0]
    indices_1d = torch.randperm(total_elements)[:n]
    
    X_train_latent_np = X_train_latent[indices_1d]
    y_train_true = y_train[indices_1d]
    X_test_latent_np = X_test_latent
    
    kmeans_model, y_train_pred, y_test_pred = kmeans_cluster(X_train_latent_np,X_test_latent_np)
    _, kmeans_cluster_mapping = majority_vote_relabeling(y_train_pred, y_train_true)

    # If there are more than one label:
    if len(np.unique(y_test_pred)) > 1:
        y_test_pred_relab =  np.vectorize(kmeans_cluster_mapping.get)(y_test_pred)
        #accuracy = accuracy_score(y_test_true, y_test_pred_relab)
    else:
        y_test_pred_relab = y_test_pred
        #accuracy = 0.5

    accuracy = accuracy_score(y_test_true, y_test_pred_relab)

    ## Set sizes
    prediction_sets, y_calib_pred_relab, y_val_pred_relab, res_metrics = \
    conformal_prediction_kmeans_with_synthetic_data(kmeans_model, (X_calib_latent,labels_calib), (X_test_latent,y_test_true), 
                                                    kmeans_cluster_mapping, alpha=0.1, print_stats=False)

    ##############################   Results
    list_q_hat.append(res_metrics[0])
    list_emp_marginal_coverage.append(res_metrics[1])
    list_avg_pred_set_size.append(res_metrics[2])
    list_set_sizes_per_class.append(res_metrics[3])
    list_cc_per_class.append(res_metrics[4])
    aris.append(adjusted_rand_score(y_test_true, y_test_pred_relab))
    nmis.append(normalized_mutual_info_score(y_test_true, y_test_pred_relab, average_method='geometric'))
    v_measures.append(v_measure_score(y_test_true, y_test_pred_relab))
    accuracys.append(accuracy_score(y_test_true, y_test_pred_relab))


results = {
    "list_q_hat" : list_q_hat,
    "list_emp_marginal_coverage": list_emp_marginal_coverage,
    "list_avg_pred_set_size": list_avg_pred_set_size,
    "list_set_sizes_per_class": list_set_sizes_per_class,
    "list_cc_per_class": list_cc_per_class,
    "list_ari": aris,
    "list_nmi": nmis,
    "list_v_measure": v_measures,
    "list_accuracy": accuracys
}


results_kmeans_ethics_df = pd.DataFrame(results)

#print(results_df)

### KNN

In [170]:
###### Use the same AE approach as KMeans

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
basic_ae = train_autoencoder_for_model_comparison(X_train, latent_dim=128, device = device, epochs=10)

X_train_latent = basic_ae.encoder(X_train.to(device)).detach().cpu().numpy()
print(X_train_latent.shape)
X_test_latent = basic_ae.encoder(X_test.to(device)).detach().cpu().numpy()
print(X_test_latent.shape)
X_calib_latent = basic_ae.encoder(embeddings_calib.to(device)).detach().cpu().numpy()
print(X_calib_latent.shape)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import importlib


def knn_cluster(X_train, y_train, X_test):
    # k-NN is trained on the labeled data (X_train, y_train)
    k_neighbors = 5 # Standard choice for k
    knn_classifier = KNeighborsClassifier(n_neighbors=k_neighbors)
    knn_classifier.fit(X_train, y_train) # Training utilizes y_train
    y_train_pred = knn_classifier.predict(X_train)
    y_test_pred = knn_classifier.predict(X_test)
    return knn_classifier, y_train_pred, y_test_pred
    

# 10000 = 14246
# 5000 = 7124
# 1000 = 1426
# 100 = 144
######

y_test_true = y_test    # Ethics

aris, nmis, v_measures, accuracys = [], [], [], []
list_q_hat, list_emp_marginal_coverage, list_avg_pred_set_size, list_set_sizes_per_class, list_cc_per_class = [], [], [], [], []
NUM_ITERS = 10
kmeans_cluster_mapping = None
for i in range(0, NUM_ITERS):
    # Random selections
    total_elements = X_train_latent.shape[0]
    n = 2000
    indices_1d = torch.randperm(total_elements)[:n]
    
    X_train_latent_np = X_train_latent[indices_1d]
    y_train_true = y_train[indices_1d]
    X_test_latent_np = X_test_latent
    y_test_true_filt = y_test_true
    
    knn_model, y_train_pred, y_test_pred = knn_cluster(X_train_latent_np,y_train_true,X_test_latent_np)
    accuracy = accuracy_score(y_test_true_filt, y_test_pred)

    ## Set sizes
    prediction_sets, y_calib_pred_relab, y_val_pred_relab, res_metrics = \
    conformal_prediction_knn_with_synthetic_data(knn_model, (X_calib_latent,labels_calib), (X_test_latent,y_test_true), 
                                                 alpha=0.1, print_stats=False)

    ##############################   Results
    list_q_hat.append(res_metrics[0])
    list_emp_marginal_coverage.append(res_metrics[1])
    list_avg_pred_set_size.append(res_metrics[2])
    list_set_sizes_per_class.append(res_metrics[3])
    list_cc_per_class.append(res_metrics[4])
    aris.append(adjusted_rand_score(y_test_true_filt, y_test_pred))
    nmis.append(normalized_mutual_info_score(y_test_true_filt, y_test_pred, average_method='geometric'))
    v_measures.append(v_measure_score(y_test_true_filt, y_test_pred))
    accuracys.append(accuracy_score(y_test_true_filt, y_test_pred))


results = {
    "list_q_hat" : list_q_hat,
    "list_emp_marginal_coverage": list_emp_marginal_coverage,
    "list_avg_pred_set_size": list_avg_pred_set_size,
    "list_set_sizes_per_class": list_set_sizes_per_class,
    "list_cc_per_class": list_cc_per_class,
    "list_ari": aris,
    "list_nmi": nmis,
    "list_v_measure": v_measures,
    "list_accuracy": accuracys
}


knn_results_df = pd.DataFrame(results)


## Ablation Study

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from pymilvus import MilvusClient, FieldSchema, CollectionSchema, DataType
from importlib import reload
from model.datasets import MilvusDataset
from model.loss import LatentPairwiseContrastiveLoss, LatentPairwiseContrastiveSAELoss, SupervisedInfoNCELoss
from experiments.sbert_contrastive_dataset import setup_ablation_study,fetch_subset,extract_from_ablation_dataloader

### Dataset

In [ ]:
DB_NAME = 'word_sense_embeddings'
BATCH_SIZE = 64
SHUFFLE = False # Shuffle data every epoch
NUM_WORKERS = 0 # Use multi-process data loading (optional, set to 0 for single process)

COLLECTION_NAME = human_values_col_name
collection_a_dataset = MilvusDataset(
    db_name=DB_NAME,
    collection_name=COLLECTION_NAME,
    batch_size = BATCH_SIZE,
    output_fields = ["id", "sense_embedding", "is_human_value", "word"],
    items2exclude = words_valid + words_calib
)
collection_a_dataloader = DataLoader(
    collection_a_dataset,
    batch_size=None,
    shuffle=SHUFFLE,
    num_workers=NUM_WORKERS
)
COLLECTION_NAME = non_values_col_name
collection_b_dataset = MilvusDataset(
    db_name=DB_NAME,
    collection_name=COLLECTION_NAME,
    batch_size = BATCH_SIZE,
    output_fields = ["id", "sense_embedding", "is_human_value", "word"],
    items2exclude = words_valid + words_calib
)
collection_b_dataloader = DataLoader(
    collection_b_dataset,
    batch_size=None,
    shuffle=SHUFFLE,
    num_workers=NUM_WORKERS
)

### Pretrained Model

In [8]:
BATCH_SIZE=64
human_values_col_name='sbert_wordnet_human_values_sense_embeddings_allminilm'
non_values_col_name='sbert_wordnet_non_values_sense_embeddings_allminilm'

milvus_dataset = setup_ablation_study(num_pairs_per_type=None,batch_size=BATCH_SIZE,
                                    train_part=0.8,val_part=0.1,
                                    human_values_col_name=human_values_col_name,
                                    non_values_col_name=non_values_col_name)

Total entities: 1865
Number of train entities: 1492
Number of validation entities: 186
Number of calibration entities: 187
1865 = 1492 + 186 + 187 = 1865


In [ ]:
# Training data
train_dataset_a_dataloader,train_dataset_b_dataloader=fetch_subset(milvus_dataset, 'train')
words_a_train, embeddings_a_train, labels_a_train = extract_from_ablation_dataloader(train_dataset_a_dataloader)
words_b_train, embeddings_b_train, labels_b_train = extract_from_ablation_dataloader(train_dataset_b_dataloader)

words_train = words_a_train + words_b_train
embeddings_train = torch.vstack((embeddings_a_train, embeddings_b_train))
labels_train = labels_a_train + labels_b_train

print(f"Number of words in training set: {len(words_train)}")
print(f"Shape of embeddings in training set: {embeddings_train.shape}")
print(f"Number of labels in training set: {len(labels_train)}")

print(f"Sample item in word list: {words_train[0]}")

In [ ]:
## Validation data
validation_dataset_a_dataloader,validation_dataset_b_dataloader=fetch_subset(milvus_dataset, 'validation')
words_a_valid, embeddings_a_valid, labels_a_valid = extract_from_ablation_dataloader(validation_dataset_a_dataloader)
words_b_valid, embeddings_b_valid, labels_b_valid = extract_from_ablation_dataloader(validation_dataset_b_dataloader)

words_valid = words_a_valid + words_b_valid
embeddings_valid = torch.vstack((embeddings_a_valid, embeddings_b_valid))
labels_valid = labels_a_valid + labels_b_valid

print(f"Number of words in validation set: {len(words_valid)}")
print(f"Shape of embeddings in validation set: {embeddings_valid.shape}")
print(f"Number of labels in validation set: {len(labels_valid)}")

print(f"Sample item in word list: {words_valid[0]}")

In [ ]:
## Calibration data
calib_dataset_a_dataloader,calib_dataset_b_dataloader = fetch_subset(milvus_dataset, 'calibration')
words_a_calib, embeddings_a_calib, labels_a_calib = extract_from_ablation_dataloader(calib_dataset_a_dataloader)
words_b_calib, embeddings_b_calib, labels_b_calib = extract_from_ablation_dataloader(calib_dataset_b_dataloader)

words_calib = words_a_calib + words_b_calib
embeddings_calib = torch.vstack((embeddings_a_calib, embeddings_b_calib))
labels_calib = labels_a_calib + labels_b_calib

print(f"Number of words in calibration set: {len(words_calib)}")
print(f"Shape of embeddings in calibration set: {embeddings_calib.shape}")
print(f"Number of labels in calibration set: {len(labels_calib)}")

print(f"Sample item in word list: {words_calib[0]}")

In [ ]:
from model.models import ContrastiveClusterLearner
#from experiments.ablation_training import EmbeddingsAutoencoderServiceAblation
from sentence_transformers import SentenceTransformer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
## Wrapper model
sbert_model_name = 'all-MiniLM-L6-v2' 
sbert_emb_model = SentenceTransformer(sbert_model_name)

ccl_model = ContrastiveClusterLearner(embed_transformer = sbert_emb_model, device=DEVICE)


In [ ]:
import random

## To align labels, centroids must be initialized by randomly selected embeddings whose labels are 0 and 1
random_index_0 = len(labels_a_train) + random.randrange(len(labels_b_train))
random_index_1 = random.randrange(len(labels_a_train))
print(f"Randomly selected indices : {random_index_0}, {random_index_1}")

## Fit to training data
pred_clusters_train, train_centroids = ccl_model.cosine_kmeans_fit(embeddings_train,num_clusters=2,initial_indices=[random_index_0,random_index_1])
pred_clusters_train = pred_clusters_train.cpu() 

# Predict centers
pred_clusters_valid = ccl_model.predict_clusters(embeddings_valid.detach().cpu(), train_centroids.cpu())
pred_clusters_calib = ccl_model.predict_clusters(embeddings_calib.detach().cpu(), train_centroids.cpu())


pred_clusters_train_calib = np.concat((pred_clusters_train, pred_clusters_calib))
pred_clusters_train_calib_valid = np.concat((pred_clusters_train_calib, pred_clusters_valid))

print(f"Number of assigned cluster centroids during training: {len(pred_clusters_train)}")
print(f"Number of assigned validation cluster centroids: {len(pred_clusters_valid)}")


In [18]:
#from sklearn.metrics import adjusted_rand_score, accuracy_score, normalized_mutual_info_score, v_measure_score

#accuracy_score(labels_valid,pred_clusters_valid)

In [ ]:

## (1) Predict categories for calibration data
pred_labels_calib = ccl_model.predict_clusters(embeddings_calib.detach().cpu(), train_centroids.cpu())

# (2) Calculate Residuals (aka conformity score) using the calibration data for the predicted labels
calib_deltas = ccl_model.calculate_calibration_conformity_scores_loo(embeddings_calib, pred_labels_calib.detach().clone())

# (3) Compute per class quantile thresholds
q_hat = {}
CONFIDENCE = 0.9
gamma = 1 - CONFIDENCE # Significance level
    
for c, deltas in calib_deltas.items():
    if not deltas: continue
    # The quantile index is calculated as ceil((n+1) * (1-confidence)) - 1
    n = len(deltas)
    idx = int(np.ceil((n + 1) * gamma)) - 1
    
    # Sort and pick the quantile
    sorted_deltas = sorted(deltas)
    #q_hat[c] = sorted_alphas[min(idx, n - 1)] # Ensure index is valid
    q_hat[c] = np.percentile(sorted_deltas, CONFIDENCE * 100)
    print(f"Predicted Class {c}: n={n}, Quantile q_hat = {q_hat[c]:.4f}")

# (4) Predict categories for validation data
pred_labels_valid = ccl_model.predict_clusters(embeddings_valid.detach().cpu(), train_centroids.cpu())

# (5) Conformity score for the validation data
all_test_scores = ccl_model.calculate_test_conformity_scores(embeddings_calib, pred_labels_calib.detach().clone(), embeddings_valid, pred_labels_valid.detach().clone())

# (6) Determine Prediction Sets for validation data
prediction_sets = []
coverage_count = 0
total_sets = len(labels_valid)
unique_classes = np.unique(labels_valid)
class_coverage_total = {c: 0 for c in unique_classes}
coverage_gaps = {}

for i in range(total_sets):
    current_true_label = labels_valid[i]
    
    # Scores for the current test sample
    scores_i = all_test_scores[i] 
    
    # Find the set of possible labels (C(x))
    C_x = ccl_model.compute_prediction_set(scores_i, q_hat)
    prediction_sets.append(C_x)
    
    if current_true_label in C_x:
        coverage_count += 1
        class_coverage_total[current_true_label] += 1
    

In [ ]:
## Evaluation
coverage = coverage_count / total_sets
avg_set_size = np.mean([len(s) for s in prediction_sets])
prediction_set_sizes = [len(s) for s in prediction_sets]
    
print("\n--- Conformal Prediction Results ---")
print(f"Target Confidence (1-alpha): {CONFIDENCE:.2f}")
print(f"Empirical Coverage: {coverage:.4f}")
print(f"Average Prediction Set Size: {avg_set_size:.3f}")

# Example visualization of the first 5 predictions
print("\nExample Predictions (True Label | Prediction Set):")
for i in range(5):
    print(f"  {labels_valid[i]} | {prediction_sets[i]}")
    
print("\nThis method successfully uses the L2-normalized Euclidean distance (equivalent to cosine distance) to build the non-conformity scores.")


In [ ]:
precision_efficiency = 1 / avg_set_size if avg_set_size > 0 else 0
    
# 4. F1-Score (Harmonic mean of Efficiency and Coverage)
# F1 = 2 * (P * R) / (P + R)
recall_coverage = coverage 
if (precision_efficiency + recall_coverage) > 0:
    f1_score = 2 * (precision_efficiency * recall_coverage) / (precision_efficiency + recall_coverage)
else:
    f1_score = 0.0

print("\n--- Conformal Prediction Results ---")
print(f"Target Confidence (1-alpha): {CONFIDENCE:.2f}")

print("\n--- Summary Metrics (Adapted for Set Prediction) ---")
print(f"Empirical Coverage (Recall): {recall_coverage:.4f}")
print(f"Average Set Size (Inverse Precision): {avg_set_size:.3f}")
print(f"Set Efficiency (Precision): {precision_efficiency:.4f}")
print(f"F1-Score: {f1_score:.4f}")

In [ ]:
# Initialize structures for Conditional Coverage tracking
prediction_sets = []
coverage_count = 0
total_sets = len(labels_valid)
unique_classes = np.unique(labels_valid)
conditional_coverage_total = {c: 0 for c in unique_classes}
conditional_coverage_correct = {c: 0 for c in unique_classes}


for i in range(total_sets):
    current_true_label = labels_valid[i]
    scores_i = all_test_scores[i] 
    
    # Find the set of possible labels (C(x))
    C_x = ccl_model.compute_prediction_set(scores_i, q_hat)
    prediction_sets.append(C_x)
    
    # Update conditional coverage tracking
    conditional_coverage_total[current_true_label] += 1

    if current_true_label in C_x:
        coverage_count += 1 # Marginal Coverage
        conditional_coverage_correct[current_true_label] += 1
        
# --- 3.4. Evaluation ---

coverage = coverage_count / total_sets


set_sizes_0, set_sizes_1, set_sizes_2 = 0, 0, 0
set_sizes = {'0':set_sizes_0, '1':set_sizes_1, '2':set_sizes_2}
             
for s in prediction_sets:
    if len(s) == 0:
       set_sizes['0'] = set_sizes['0'] + 1
    elif len(s) == 1:
       set_sizes['1'] = set_sizes['1'] + 1
    if len(s) == 2:
       set_sizes['2'] = set_sizes['2'] + 1

total_set_size = sum(set_sizes.values())
print(set_sizes)
avg_set_size = np.mean([len(s) for s in prediction_sets])
coverage_gaps = {}

print("\n--- Conformal Prediction Results ---")
print(f"Target Confidence (1-alpha): {CONFIDENCE:.2f}")
print(f"Empirical Marginal Coverage: {coverage:.4f}")
print(f"Average Prediction Set Size: {avg_set_size:.3f}")

for k,v in set_sizes.items():
    proportion = int(v)/total_set_size
    print(f"Set size={k} Proportion={proportion:.3f}")

print("\n--- Empirical Conditional Coverage ---")
# This shows the actual coverage achieved for each specific class
for c in sorted(conditional_coverage_total.keys()):
    total = conditional_coverage_total[c]
    correct = conditional_coverage_correct[c]
    if total > 0:
        cond_cov = correct / total     # Matches the cc cov definition in the reference paper (Ding et al)
        print(f"  Class {c} Coverage (n={total}): {cond_cov:.4f}")
    else:
        print(f"  Class {c} Coverage (n=0): N/A (No test samples for this class)")

    gap = (cond_cov - CONFIDENCE) # Signed gap
    coverage_gaps[c] = {
        'class_conditional_coverage': f"{cond_cov:.3f}",
        'gap': f"{gap:.3f}",
        'absolute_gap': f"{abs(gap):.3f}"
    }

print("\n--- Class Conditional Coverage Gaps ---")  
total_cov_gap = 0
for c in sorted(coverage_gaps.keys()):
    cov_gap = coverage_gaps[c]
    print(cov_gap)
    print(f"Cluster {c}: Class Conditional={cov_gap['class_conditional_coverage']}, Gap={cov_gap['gap']}")
    total_cov_gap = total_cov_gap + float(cov_gap['absolute_gap'])


print("\n--- Avg Class Conditional Coverage Gap ---") 
avg_cov_gap = (100 * total_cov_gap)/(len(conditional_coverage_total.keys()))
print(f"{avg_cov_gap:.2f}%")
    

### Fine-tuned SBERT model with SBERT-generated original embeddings

In [ ]:
NUM_PAIRS=10  
BATCH_SIZE=64
human_values_col_name='sbert_wordnet_human_values_sense_embeddings_allminilm_ft'
non_values_col_name='sbert_wordnet_non_values_sense_embeddings_allminilm_ft'

contrastive_train_dataloader,train_labels,milvus_dataset = setup_ablation_study(num_pairs_per_type=NUM_PAIRS,batch_size=BATCH_SIZE,
                                    train_part=0.8,val_part=0.1,
                                    human_values_col_name=human_values_col_name,
                                    non_values_col_name=non_values_col_name)

In [3]:
contrastive_data = {
        "text1": [],
        "text2": [],
        "label": []
    }

for batch in contrastive_train_dataloader:
    # Extract text columns (usually lists of strings in a batch)
    contrastive_data["text1"].extend(batch["text1"])
    contrastive_data["text2"].extend(batch["text2"])

    # Extract labels
    labels = batch["label"]

    # If labels are PyTorch Tensors, move to CPU and convert to list
    if torch.is_tensor(labels):
        contrastive_data["label"].extend(labels.cpu().tolist())
    else:
        contrastive_data["label"].extend(labels)


contrastive_df = pd.DataFrame(contrastive_data)


In [ ]:
# Training data
train_dataset_a_dataloader,train_dataset_b_dataloader=fetch_subset(milvus_dataset, 'train')
words_a_train, embeddings_a_train, labels_a_train = extract_from_ablation_dataloader(train_dataset_a_dataloader)
words_b_train, embeddings_b_train, labels_b_train = extract_from_ablation_dataloader(train_dataset_b_dataloader)

words_train = words_a_train + words_b_train
embeddings_train = torch.vstack((embeddings_a_train, embeddings_b_train))
labels_train = labels_a_train + labels_b_train

print(f"Number of words in training set: {len(words_train)}")
print(f"Shape of embeddings in training set: {embeddings_train.shape}")
print(f"Number of labels in training set: {len(labels_train)}")

print(f"Sample item in word list: {words_train[0]}")

In [ ]:
## Validation data
validation_dataset_a_dataloader,validation_dataset_b_dataloader=fetch_subset(milvus_dataset, 'validation')
words_a_valid, embeddings_a_valid, labels_a_valid = extract_from_ablation_dataloader(validation_dataset_a_dataloader)
words_b_valid, embeddings_b_valid, labels_b_valid = extract_from_ablation_dataloader(validation_dataset_b_dataloader)

words_valid = words_a_valid + words_b_valid
embeddings_valid = torch.vstack((embeddings_a_valid, embeddings_b_valid))
labels_valid = labels_a_valid + labels_b_valid

print(f"Number of words in validation set: {len(words_valid)}")
print(f"Shape of embeddings in validation set: {embeddings_valid.shape}")
print(f"Number of labels in validation set: {len(labels_valid)}")

print(f"Sample item in word list: {words_valid[0]}")

In [ ]:
## Calibration data
calib_dataset_a_dataloader,calib_dataset_b_dataloader = fetch_subset(milvus_dataset, 'calibration')
words_a_calib, embeddings_a_calib, labels_a_calib = extract_from_ablation_dataloader(calib_dataset_a_dataloader)
words_b_calib, embeddings_b_calib, labels_b_calib = extract_from_ablation_dataloader(calib_dataset_b_dataloader)

words_calib = words_a_calib + words_b_calib
embeddings_calib = torch.vstack((embeddings_a_calib, embeddings_b_calib))
labels_calib = labels_a_calib + labels_b_calib

print(f"Number of words in calibration set: {len(words_calib)}")
print(f"Shape of embeddings in calibration set: {embeddings_calib.shape}")
print(f"Number of labels in calibration set: {len(labels_calib)}")

print(f"Sample item in word list: {words_calib[0]}")

In [23]:
## TRAIN THE MODEL IN THE BACKEND BECAUSE OF LIBRARY DEPENDENCIES (sbert_finetune.py)

In [29]:
from sentence_transformers import SentenceTransformer

# Load the finetuned model
input_path = '/home/jovyan/sbert_cache/saved/all-MiniLM-L6-v2'

# Load the model from the local directory
loaded_model = SentenceTransformer(input_path)

In [30]:
from model.models import ContrastiveClusterLearner
#from experiments.ablation_training import EmbeddingsAutoencoderServiceAblation
from sentence_transformers import SentenceTransformer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ccl_model = ContrastiveClusterLearner(embed_transformer = loaded_model, device=DEVICE)

In [ ]:
import random

## To align labels, centroids must be initialized by randomly selected embeddings whose labels are 0 and 1
random_index_0 = len(labels_a_train) + random.randrange(len(labels_b_train))
random_index_1 = random.randrange(len(labels_a_train))
print(f"Randomly selected indices : {random_index_0}, {random_index_1}")

## Fit to training data
pred_clusters_train, train_centroids = ccl_model.cosine_kmeans_fit(embeddings_train,num_clusters=2,initial_indices=[random_index_0,random_index_1])
pred_clusters_train = pred_clusters_train.cpu() 

# Predict centers
pred_clusters_valid = ccl_model.predict_clusters(embeddings_valid.detach().cpu(), train_centroids.cpu())
pred_clusters_calib = ccl_model.predict_clusters(embeddings_calib.detach().cpu(), train_centroids.cpu())

pred_clusters_train_calib = np.concat((pred_clusters_train, pred_clusters_calib))
pred_clusters_train_calib_valid = np.concat((pred_clusters_train_calib, pred_clusters_valid))

print(f"Number of assigned cluster centroids during training: {len(pred_clusters_train)}")
print(f"Number of assigned validation cluster centroids: {len(pred_clusters_valid)}")

In [ ]:
from sklearn.metrics import adjusted_rand_score, accuracy_score, normalized_mutual_info_score, v_measure_score

accuracy_score(labels_valid,pred_clusters_valid)

In [ ]:

## (1) Predict categories for calibration data
pred_labels_calib = ccl_model.predict_clusters(embeddings_calib.detach().cpu(), train_centroids.cpu())

# (2) Calculate Residuals (aka conformity score) using the calibration data for the predicted labels
calib_deltas = ccl_model.calculate_calibration_conformity_scores_loo(embeddings_calib, pred_labels_calib.detach().clone())

# (3) Compute per class quantile thresholds
q_hat = {}
CONFIDENCE = 0.9
gamma = 1 - CONFIDENCE # Significance level
    
for c, deltas in calib_deltas.items():
    if not deltas: continue
    # The quantile index is calculated as ceil((n+1) * (1-confidence)) - 1
    n = len(deltas)
    idx = int(np.ceil((n + 1) * gamma)) - 1
    
    # Sort and pick the quantile
    sorted_deltas = sorted(deltas)
    #q_hat[c] = sorted_alphas[min(idx, n - 1)] # Ensure index is valid
    q_hat[c] = np.percentile(sorted_deltas, CONFIDENCE * 100)
    print(f"Predicted Class {c}: n={n}, Quantile q_hat = {q_hat[c]:.4f}")

# (4) Predict categories for validation data
pred_labels_valid = ccl_model.predict_clusters(embeddings_valid.detach().cpu(), train_centroids.cpu())

# (5) Conformity score for the validation data
all_test_scores = ccl_model.calculate_test_conformity_scores(embeddings_calib, pred_labels_calib.detach().clone(), embeddings_valid, pred_labels_valid.detach().clone())

# (6) Determine Prediction Sets for validation data
prediction_sets = []
coverage_count = 0
total_sets = len(labels_valid)
unique_classes = np.unique(labels_valid)
class_coverage_total = {c: 0 for c in unique_classes}
coverage_gaps = {}

for i in range(total_sets):
    current_true_label = labels_valid[i]
    
    # Scores for the current test sample
    scores_i = all_test_scores[i] 
    
    # Find the set of possible labels (C(x))
    C_x = ccl_model.compute_prediction_set(scores_i, q_hat)
    prediction_sets.append(C_x)
    
    if current_true_label in C_x:
        coverage_count += 1
        class_coverage_total[current_true_label] += 1
    

In [ ]:
## Evaluation
coverage = coverage_count / total_sets
avg_set_size = np.mean([len(s) for s in prediction_sets])
prediction_set_sizes = [len(s) for s in prediction_sets]
    
print("\n--- Conformal Prediction Results ---")
print(f"Target Confidence (1-alpha): {CONFIDENCE:.2f}")
print(f"Empirical Coverage: {coverage:.4f}")
print(f"Average Prediction Set Size: {avg_set_size:.3f}")

# Example visualization of the first 5 predictions
print("\nExample Predictions (True Label | Prediction Set):")
for i in range(5):
    print(f"  {labels_valid[i]} | {prediction_sets[i]}")
    

In [ ]:
precision_efficiency = 1 / avg_set_size if avg_set_size > 0 else 0
    
# 4. F1-Score (Harmonic mean of Efficiency and Coverage)
# F1 = 2 * (P * R) / (P + R)
recall_coverage = coverage 
if (precision_efficiency + recall_coverage) > 0:
    f1_score = 2 * (precision_efficiency * recall_coverage) / (precision_efficiency + recall_coverage)
else:
    f1_score = 0.0

print("\n--- Conformal Prediction Results ---")
print(f"Target Confidence (1-alpha): {CONFIDENCE:.2f}")

print("\n--- Summary Metrics (Adapted for Set Prediction) ---")
print(f"Empirical Coverage (Recall): {recall_coverage:.4f}")
print(f"Average Set Size (Inverse Precision): {avg_set_size:.3f}")
print(f"Set Efficiency (Precision): {precision_efficiency:.4f}")
print(f"F1-Score: {f1_score:.4f}")

In [ ]:
# Initialize structures for Conditional Coverage tracking
prediction_sets = []
coverage_count = 0
total_sets = len(labels_valid)
unique_classes = np.unique(labels_valid)
conditional_coverage_total = {c: 0 for c in unique_classes}
conditional_coverage_correct = {c: 0 for c in unique_classes}

for i in range(total_sets):
    current_true_label = labels_valid[i]
    scores_i = all_test_scores[i] 
    
    # Find the set of possible labels (C(x))
    C_x = ccl_model.compute_prediction_set(scores_i, q_hat)
    prediction_sets.append(C_x)
    
    # Update conditional coverage tracking
    conditional_coverage_total[current_true_label] += 1

    if current_true_label in C_x:
        coverage_count += 1 # Marginal Coverage
        conditional_coverage_correct[current_true_label] += 1
        
# --- 3.4. Evaluation ---
coverage = coverage_count / total_sets
set_sizes_0, set_sizes_1, set_sizes_2 = 0, 0, 0
set_sizes = {'0':set_sizes_0, '1':set_sizes_1, '2':set_sizes_2}
             
for s in prediction_sets:
    if len(s) == 0:
       set_sizes['0'] = set_sizes['0'] + 1
    elif len(s) == 1:
       set_sizes['1'] = set_sizes['1'] + 1
    if len(s) == 2:
       set_sizes['2'] = set_sizes['2'] + 1

total_set_size = sum(set_sizes.values())
print(set_sizes)
avg_set_size = np.mean([len(s) for s in prediction_sets])
coverage_gaps = {}

print("\n--- Conformal Prediction Results ---")
print(f"Target Confidence (1-alpha): {CONFIDENCE:.2f}")
print(f"Empirical Marginal Coverage: {coverage:.4f}")
print(f"Average Prediction Set Size: {avg_set_size:.3f}")

for k,v in set_sizes.items():
    proportion = int(v)/total_set_size
    print(f"Set size={k} Proportion={proportion:.3f}")

print("\n--- Empirical Conditional Coverage ---")
# This shows the actual coverage achieved for each specific class
for c in sorted(conditional_coverage_total.keys()):
    total = conditional_coverage_total[c]
    correct = conditional_coverage_correct[c]
    if total > 0:
        cond_cov = correct / total     # Matches the cc cov definition in the reference paper (Ding et al)
        print(f"  Class {c} Coverage (n={total}): {cond_cov:.4f}")
    else:
        print(f"  Class {c} Coverage (n=0): N/A (No test samples for this class)")

    gap = (cond_cov - CONFIDENCE) # Signed gap
    coverage_gaps[c] = {
        'class_conditional_coverage': f"{cond_cov:.3f}",
        'gap': f"{gap:.3f}",
        'absolute_gap': f"{abs(gap):.3f}"
    }

print("\n--- Class Conditional Coverage Gaps ---")  
total_cov_gap = 0
for c in sorted(coverage_gaps.keys()):
    cov_gap = coverage_gaps[c]
    print(cov_gap)
    print(f"Cluster {c}: Class Conditional={cov_gap['class_conditional_coverage']}, Gap={cov_gap['gap']}")
    total_cov_gap = total_cov_gap + float(cov_gap['absolute_gap'])


print("\n--- Avg Class Conditional Coverage Gap ---") 
avg_cov_gap = (100 * total_cov_gap)/(len(conditional_coverage_total.keys()))
print(f"{avg_cov_gap:.2f}%")
    

### Random projection model

In [ ]:
BATCH_SIZE=64
human_values_col_name='wordnet_human_values_sense_embeddings_qwen38b'
non_values_col_name='wordnet_non_values_sense_embeddings_qwen38b'

milvus_dataset = setup_ablation_study(num_pairs_per_type=None,batch_size=BATCH_SIZE,
                                    train_part=0.8,val_part=0.1,
                                    human_values_col_name=human_values_col_name,
                                    non_values_col_name=non_values_col_name)

In [ ]:
# Training data
train_dataset_a_dataloader,train_dataset_b_dataloader=fetch_subset(milvus_dataset, 'train')
words_a_train, embeddings_a_train, labels_a_train = extract_from_ablation_dataloader(train_dataset_a_dataloader)
words_b_train, embeddings_b_train, labels_b_train = extract_from_ablation_dataloader(train_dataset_b_dataloader)

words_train = words_a_train + words_b_train
embeddings_train = torch.vstack((embeddings_a_train, embeddings_b_train))
labels_train = labels_a_train + labels_b_train

print(f"Number of words in training set: {len(words_train)}")
print(f"Shape of embeddings in training set: {embeddings_train.shape}")
print(f"Number of labels in training set: {len(labels_train)}")

print(f"Sample item in word list: {words_train[0]}")

In [ ]:
## Validation data
validation_dataset_a_dataloader,validation_dataset_b_dataloader=fetch_subset(milvus_dataset, 'validation')
words_a_valid, embeddings_a_valid, labels_a_valid = extract_from_ablation_dataloader(validation_dataset_a_dataloader)
words_b_valid, embeddings_b_valid, labels_b_valid = extract_from_ablation_dataloader(validation_dataset_b_dataloader)

words_valid = words_a_valid + words_b_valid
embeddings_valid = torch.vstack((embeddings_a_valid, embeddings_b_valid))
labels_valid = labels_a_valid + labels_b_valid

print(f"Number of words in validation set: {len(words_valid)}")
print(f"Shape of embeddings in validation set: {embeddings_valid.shape}")
print(f"Number of labels in validation set: {len(labels_valid)}")

print(f"Sample item in word list: {words_valid[0]}")

In [ ]:
## Calibration data
calib_dataset_a_dataloader,calib_dataset_b_dataloader = fetch_subset(milvus_dataset, 'calibration')
words_a_calib, embeddings_a_calib, labels_a_calib = extract_from_ablation_dataloader(calib_dataset_a_dataloader)
words_b_calib, embeddings_b_calib, labels_b_calib = extract_from_ablation_dataloader(calib_dataset_b_dataloader)

words_calib = words_a_calib + words_b_calib
embeddings_calib = torch.vstack((embeddings_a_calib, embeddings_b_calib))
labels_calib = labels_a_calib + labels_b_calib

print(f"Number of words in calibration set: {len(words_calib)}")
print(f"Shape of embeddings in calibration set: {embeddings_calib.shape}")
print(f"Number of labels in calibration set: {len(labels_calib)}")

print(f"Sample item in word list: {words_calib[0]}")

In [13]:
from sklearn.random_projection import SparseRandomProjection
from sklearn.random_projection import GaussianRandomProjection

transformer1 = SparseRandomProjection(random_state=42, n_components=256)
#transformer2 = SparseRandomProjection(random_state=42, eps=0.01)    # DOES NOT WORK. unspecified n_components led to a target dimension of 588715 which is larger than the original space with n_features=4096

transformer_gr = SparseRandomProjection(random_state=42, eps=0.9) 

In [10]:
### (1) Specified n_components

embeddings_train_np = embeddings_train.detach().cpu().numpy()
embeddings_train_np_red = transformer1.fit_transform(embeddings_train_np)

In [14]:
## Gaussian random

embeddings_train_np = embeddings_train.detach().cpu().numpy()
embeddings_train_np_red = transformer_gr.fit_transform(embeddings_train_np)


In [15]:
from model.models import ContrastiveClusterLearner
#from experiments.ablation_training import EmbeddingsAutoencoderServiceAblation
from sentence_transformers import SentenceTransformer

DEVICE = "cpu"
## Wrapper model
ccl_model = ContrastiveClusterLearner(embed_transformer = transformer_gr, device=DEVICE)


In [ ]:
import random

## To align labels, centroids must be initialized by randomly selected embeddings whose labels are 0 and 1
random_index_0 = len(labels_a_train) + random.randrange(len(labels_b_train))
random_index_1 = random.randrange(len(labels_a_train))
print(f"Randomly selected indices : {random_index_0}, {random_index_1}")

## Fit to training data
pred_clusters_train, train_centroids = ccl_model.cosine_kmeans_fit(embeddings_train,num_clusters=2,initial_indices=[random_index_0,random_index_1])
pred_clusters_train = pred_clusters_train.cpu() 

# Predict centers
pred_clusters_valid = ccl_model.predict_clusters(embeddings_valid.detach().cpu(), train_centroids.cpu())
pred_clusters_calib = ccl_model.predict_clusters(embeddings_calib.detach().cpu(), train_centroids.cpu())

pred_clusters_train_calib = np.concat((pred_clusters_train, pred_clusters_calib))
pred_clusters_train_calib_valid = np.concat((pred_clusters_train_calib, pred_clusters_valid))

print(f"Number of assigned cluster centroids during training: {len(pred_clusters_train)}")
print(f"Number of assigned validation cluster centroids: {len(pred_clusters_valid)}")

In [21]:
#from sklearn.metrics import adjusted_rand_score, accuracy_score, normalized_mutual_info_score, v_measure_score

#accuracy_score(labels_valid,pred_clusters_valid)

In [ ]:

## (1) Predict categories for calibration data
pred_labels_calib = ccl_model.predict_clusters(embeddings_calib.detach().cpu(), train_centroids.cpu())

# (2) Calculate Residuals (aka conformity score) using the calibration data for the predicted labels
calib_deltas = ccl_model.calculate_calibration_conformity_scores_loo(embeddings_calib, pred_labels_calib.detach().clone())

# (3) Compute per class quantile thresholds
q_hat = {}
CONFIDENCE = 0.9
gamma = 1 - CONFIDENCE # Significance level
    
for c, deltas in calib_deltas.items():
    if not deltas: continue
    # The quantile index is calculated as ceil((n+1) * (1-confidence)) - 1
    n = len(deltas)
    idx = int(np.ceil((n + 1) * gamma)) - 1
    
    # Sort and pick the quantile
    sorted_deltas = sorted(deltas)
    #q_hat[c] = sorted_alphas[min(idx, n - 1)] # Ensure index is valid
    q_hat[c] = np.percentile(sorted_deltas, CONFIDENCE * 100)
    print(f"Predicted Class {c}: n={n}, Quantile q_hat = {q_hat[c]:.4f}")

# (4) Predict categories for validation data
pred_labels_valid = ccl_model.predict_clusters(embeddings_valid.detach().cpu(), train_centroids.cpu())

# (5) Conformity score for the validation data
all_test_scores = ccl_model.calculate_test_conformity_scores(embeddings_calib, pred_labels_calib.detach().clone(), embeddings_valid, pred_labels_valid.detach().clone())

# (6) Determine Prediction Sets for validation data
prediction_sets = []
coverage_count = 0
total_sets = len(labels_valid)
unique_classes = np.unique(labels_valid)
class_coverage_total = {c: 0 for c in unique_classes}
coverage_gaps = {}

for i in range(total_sets):
    current_true_label = labels_valid[i]
    
    # Scores for the current test sample
    scores_i = all_test_scores[i] 
    
    # Find the set of possible labels (C(x))
    C_x = ccl_model.compute_prediction_set(scores_i, q_hat)
    prediction_sets.append(C_x)
    
    if current_true_label in C_x:
        coverage_count += 1
        class_coverage_total[current_true_label] += 1
    

In [ ]:
## Evaluation
coverage = coverage_count / total_sets
avg_set_size = np.mean([len(s) for s in prediction_sets])
prediction_set_sizes = [len(s) for s in prediction_sets]
    
print("\n--- Conformal Prediction Results ---")
print(f"Target Confidence (1-alpha): {CONFIDENCE:.2f}")
print(f"Empirical Coverage: {coverage:.4f}")
print(f"Average Prediction Set Size: {avg_set_size:.3f}")

# Example visualization of the first 5 predictions
print("\nExample Predictions (True Label | Prediction Set):")
for i in range(5):
    print(f"  {labels_valid[i]} | {prediction_sets[i]}")
    
print("\nThis method successfully uses the L2-normalized Euclidean distance (equivalent to cosine distance) to build the non-conformity scores.")


In [ ]:
precision_efficiency = 1 / avg_set_size if avg_set_size > 0 else 0
    
# 4. F1-Score (Harmonic mean of Efficiency and Coverage)
# F1 = 2 * (P * R) / (P + R)
recall_coverage = coverage 
if (precision_efficiency + recall_coverage) > 0:
    f1_score = 2 * (precision_efficiency * recall_coverage) / (precision_efficiency + recall_coverage)
else:
    f1_score = 0.0

print("\n--- Conformal Prediction Results ---")
print(f"Target Confidence (1-alpha): {CONFIDENCE:.2f}")

print("\n--- Summary Metrics (Adapted for Set Prediction) ---")
print(f"Empirical Coverage (Recall): {recall_coverage:.4f}")
print(f"Average Set Size (Inverse Precision): {avg_set_size:.3f}")
print(f"Set Efficiency (Precision): {precision_efficiency:.4f}")
print(f"F1-Score: {f1_score:.4f}")

In [ ]:
# Initialize structures for Conditional Coverage tracking
prediction_sets = []
coverage_count = 0
total_sets = len(labels_valid)
unique_classes = np.unique(labels_valid)
conditional_coverage_total = {c: 0 for c in unique_classes}
conditional_coverage_correct = {c: 0 for c in unique_classes}


for i in range(total_sets):
    current_true_label = labels_valid[i]
    scores_i = all_test_scores[i] 
    
    # Find the set of possible labels (C(x))
    C_x = ccl_model.compute_prediction_set(scores_i, q_hat)
    prediction_sets.append(C_x)
    
    # Update conditional coverage tracking
    conditional_coverage_total[current_true_label] += 1

    if current_true_label in C_x:
        coverage_count += 1 # Marginal Coverage
        conditional_coverage_correct[current_true_label] += 1
        
# --- 3.4. Evaluation ---

coverage = coverage_count / total_sets


set_sizes_0, set_sizes_1, set_sizes_2 = 0, 0, 0
set_sizes = {'0':set_sizes_0, '1':set_sizes_1, '2':set_sizes_2}
             
for s in prediction_sets:
    if len(s) == 0:
       set_sizes['0'] = set_sizes['0'] + 1
    elif len(s) == 1:
       set_sizes['1'] = set_sizes['1'] + 1
    if len(s) == 2:
       set_sizes['2'] = set_sizes['2'] + 1

total_set_size = sum(set_sizes.values())
print(set_sizes)
avg_set_size = np.mean([len(s) for s in prediction_sets])
coverage_gaps = {}

print("\n--- Conformal Prediction Results ---")
print(f"Target Confidence (1-alpha): {CONFIDENCE:.2f}")
print(f"Empirical Marginal Coverage: {coverage:.4f}")
print(f"Average Prediction Set Size: {avg_set_size:.3f}")

for k,v in set_sizes.items():
    proportion = int(v)/total_set_size
    print(f"Set size={k} Proportion={proportion:.3f}")

print("\n--- Empirical Conditional Coverage ---")
# This shows the actual coverage achieved for each specific class
for c in sorted(conditional_coverage_total.keys()):
    total = conditional_coverage_total[c]
    correct = conditional_coverage_correct[c]
    if total > 0:
        cond_cov = correct / total     # Matches the cc cov definition in the reference paper (Ding et al)
        print(f"  Class {c} Coverage (n={total}): {cond_cov:.4f}")
    else:
        print(f"  Class {c} Coverage (n=0): N/A (No test samples for this class)")

    gap = (cond_cov - CONFIDENCE) # Signed gap
    coverage_gaps[c] = {
        'class_conditional_coverage': f"{cond_cov:.3f}",
        'gap': f"{gap:.3f}",
        'absolute_gap': f"{abs(gap):.3f}"
    }

print("\n--- Class Conditional Coverage Gaps ---")  
total_cov_gap = 0
for c in sorted(coverage_gaps.keys()):
    cov_gap = coverage_gaps[c]
    print(cov_gap)
    print(f"Cluster {c}: Class Conditional={cov_gap['class_conditional_coverage']}, Gap={cov_gap['gap']}")
    total_cov_gap = total_cov_gap + float(cov_gap['absolute_gap'])


print("\n--- Avg Class Conditional Coverage Gap ---") 
avg_cov_gap = (100 * total_cov_gap)/(len(conditional_coverage_total.keys()))
print(f"{avg_cov_gap:.2f}%")
    